In [ ]:
import sqlite3
import pandas as pd
from pathlib import Path
from IPython.display import display, Markdown

DB_PATH = Path(r"D:\studia\DermaAI\Backend\DermaAPI\derma.db")

tables = ["SkinAnalysis", "AnalysisResult", "Detection", "SkinAnalysisTag"]

if DB_PATH.exists():
    conn = sqlite3.connect(DB_PATH)
    
    for table_name in tables:
        display(Markdown(f"---"))
        display(Markdown(f"## Tabela: `{table_name}`"))
        
        # 1. Pobranie i wyświetlenie schematu tabeli (Typy danych, klucze itp.)
        display(Markdown(f"### Schemat tabeli:"))
        try:
            schema_df = pd.read_sql(f'PRAGMA table_info("{table_name}")', conn)

            # cid (id kolumny), name (nazwa), type (typ np. TEXT, INTEGER), 
            # notnull (czy wymagane: 1=tak, 0=nie), dflt_value (domyślna wartość), pk (czy klucz główny)
            display(schema_df[['cid', 'name', 'type', 'notnull', 'dflt_value', 'pk']])
        except Exception as e:
            display(Markdown(f"Błąd pobierania schematu dla `{table_name}`: `{e}`"))
            

        display(Markdown(f"### Zawartość (Wartości w tabeli):"))
        try:
            query = f'SELECT * FROM "{table_name}"'
            df = pd.read_sql(query, conn)
            
            if df.empty:
                display(Markdown(f"*Tabela `{table_name}` jest pusta (brak rekordów).*"))
            else:
                display(Markdown(f"Liczba rekordów: **{len(df)}**"))
                display(df)
                
        except Exception as e:
            display(Markdown(f"Błąd podczas odczytu danych z tabeli `{table_name}`: `{e}`"))
            
    conn.close()
    display(Markdown("---"))
    display(Markdown("**Analiza bazy zakończona. Połączenie zostało zamknięte.**"))
else:
    print(f"Błąd: Nie znaleziono bazy danych pod ścieżką {DB_PATH}")

In [ ]:
conn = sqlite3.connect(DB_PATH)

query_tags = """
    SELECT 
        substr(sa.uploadDate, 1, 10) AS date,
        sat.Tag AS tag_enum,
        sat.Value AS tag_value
    FROM SkinAnalysis sa
    JOIN SkinAnalysisTag sat ON sa.Id = sat.SkinAnalysisId
    WHERE sa.userId = ?
"""
df_tags = pd.read_sql(query_tags, conn, params=(USER_ID,))
conn.close()

if df_tags.empty:
    raise RuntimeError("Nie znaleziono żadnych tagów/czynników dla tego użytkownika w bazie.")


TAG_NAME_MAPPING = {
    0: "PoczatekCyklu",
    1: "SloneJedzenie",
    2: "OstreJedzenie",
    3: "FastFood",
    4: "Alkohol"
}


df_tags["tag_enum"] = pd.to_numeric(df_tags["tag_enum"], errors="coerce").map(TAG_NAME_MAPPING)
df_tags["date"] = pd.to_datetime(df_tags["date"], errors="coerce").dt.normalize()


print("[LOG] Unikalne zmapowane tagi w df_tags:", df_tags["tag_enum"].unique())

factors_df = df_tags.pivot_table(
    index="date", 
    columns="tag_enum", 
    values="tag_value", 
    aggfunc="max" 
).reset_index()

min_date = min(analysis_df["date"].min(), factors_df["date"].min())
max_date = max(analysis_df["date"].max(), factors_df["date"].max())
all_dates_range = pd.date_range(min_date, max_date, freq="D")

# Przebudowanie factors_df, aby zawierał każdy dzień po kolei w wybranym zakresie
factors_df = factors_df.set_index("date").reindex(all_dates_range).rename_axis("date").reset_index()

for col in factors_df.columns:
    if col != "date":
        factors_df[col] = factors_df[col].fillna(0).astype(int)

FACTORS = [str(col) for col in factors_df.columns if col != "date"]

print(f"[LOG] Wykryte i aktywne czynniki (FACTORS): {FACTORS}")
display(factors_df.head(10))

In [ ]:
import sqlite3
import pandas as pd
from IPython.display import display, Markdown

conn = sqlite3.connect(DB_PATH)

query_notes = """
    SELECT 
        substr(uploadDate, 1, 10) AS date,
        notes
    FROM SkinAnalysis
    WHERE userId = ? AND notes IS NOT NULL AND notes != ''
    ORDER BY uploadDate ASC
"""
notes_df = pd.read_sql(query_notes, conn, params=(USER_ID,))
conn.close()

display(Markdown(f"### Zestawienie: Data i Notatka (Łącznie wpisów: **{len(notes_df)}**)"))
display(notes_df)

In [ ]:
import sqlite3
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown
from scipy.stats import spearmanr
from statsmodels.stats.multitest import multipletests
import seaborn as sns

plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.edgecolor'] = '#bdc3c7'
plt.rcParams['axes.linewidth'] = 0.8

DB_PATH = Path(r"D:\studia\DermaAI\Backend\DermaAPI\derma.db")
USER_ID = "1a2c5462-4439-46ce-9a44-d025be764238"

if not DB_PATH.exists():
    raise FileNotFoundError(f"Nie znaleziono bazy danych pod ścieżką: {DB_PATH.resolve()}")

print(f"[LOG] Łączenie z bazą SQLite: {DB_PATH.resolve()}")
conn = sqlite3.connect(DB_PATH)


# 1. POBIERANIE WYNIKÓW ANALIZ (SkinAnalysis + AnalysisResult)
query_analysis = """
    SELECT 
        sa.Id AS analysis_id,
        sa.imageUrl AS image_path,
        sa.uploadDate AS upload_date,
        sa.notes AS notes,
        ar.acneScore AS score,
        ar.acneCount AS acne_count,
        ar.outputImageUrl AS predicted_image_path
    FROM SkinAnalysis sa
    LEFT JOIN AnalysisResult ar ON sa.Id = ar.skinAnalysisId
    WHERE sa.userId = ?
"""
analysis_df = pd.read_sql(query_analysis, conn, params=(USER_ID,))

# Normalizacja danych
analysis_df["date"] = pd.to_datetime(analysis_df["upload_date"], errors="coerce").dt.normalize()
analysis_df["score"] = pd.to_numeric(analysis_df["score"], errors="coerce")
analysis_df["acne_count"] = pd.to_numeric(analysis_df["acne_count"], errors="coerce").fillna(0).astype(int)

# Wykrywanie strony twarzy (side) z notatek lub ścieżki
def extract_side(row):
    notes = str(row.get("notes", "")).lower()
    path_str = str(row.get("image_path", "")).lower()
    for side in ["left", "right", "front"]:
        if side in notes or side in path_str:
            return side
    return "front"

analysis_df["side"] = analysis_df.apply(extract_side, axis=1)
analysis_df = analysis_df.dropna(subset=["date", "score"]).sort_values(["date", "side"]).reset_index(drop=True)


# 2. POBIERANIE TAGÓW Z TABELI `SkinAnalysisTag`

query_tags = """
    SELECT 
        substr(sa.uploadDate, 1, 10) AS date,
        sat.Tag AS tag_enum,
        sat.Value AS tag_value
    FROM SkinAnalysis sa
    JOIN SkinAnalysisTag sat ON sa.Id = sat.SkinAnalysisId
    WHERE sa.userId = ?
"""
df_tags = pd.read_sql(query_tags, conn, params=(USER_ID,))
conn.close() 


TAG_NAME_MAPPING = {
    0: "poczatek_cyklu",
    1: "slone_jedzenie",
    2: "ostre_jedzenie",
    3: "fast_food",
    4: "alkohol"
}

df_tags["tag_enum"] = pd.to_numeric(df_tags["tag_enum"], errors="coerce").map(TAG_NAME_MAPPING).fillna(df_tags["tag_enum"].astype(str))
df_tags["date"] = pd.to_datetime(df_tags["date"], errors="coerce").dt.normalize()

factors_df = df_tags.pivot_table(
    index="date", 
    columns="tag_enum", 
    values="tag_value", 
    aggfunc="max"
).reset_index()

min_date = min(analysis_df["date"].min(), factors_df["date"].min())
max_date = max(analysis_df["date"].max(), factors_df["date"].max())
all_dates_range = pd.date_range(min_date, max_date, freq="D")

factors_df = factors_df.set_index("date").reindex(all_dates_range).rename_axis("date").reset_index()
for col in factors_df.columns:
    if col != "date":
        factors_df[col] = factors_df[col].fillna(0).astype(int)

FACTORS = [str(col) for col in factors_df.columns if col != "date"]
print(f"[LOG] Wykryte czynniki z bazy SQLite: {FACTORS}")

analysis_df = analysis_df.merge(factors_df, on="date", how="left")
for factor in FACTORS:
    analysis_df[factor] = analysis_df[factor].fillna(0).astype(int)

daily_df = (
    analysis_df.groupby("date")
    .agg(
        mean_score=("score", "mean"),
        median_score=("score", "median"),
        std_score=("score", "std"),
        total_acne_count=("acne_count", "sum"),
        n_images=("score", "count"),
    )
    .reset_index()
    .sort_values("date")
)


# 3. WYKRESY EWOLUCJI W CZASIE (AGREGACJA)
side_styles = {
    "left": {"color": "#E74C3C", "label": "Lewa strona"},
    "right": {"color": "#3498DB", "label": "Prawa strona"},
    "front": {"color": "#2ECC71", "label": "Przód"},
}

def plot_aggregated_scores(n_days=7, title=None, date_format="%Y-%m-%d"):
    if n_days < 1:
        raise ValueError("n_days musi być >= 1")

    title = title or f"Ewolucja zmian trądzikowych — agregacja co {n_days} dni"
    start_date = min(analysis_df["date"].min(), daily_df["date"].min()).normalize()

    def aggregate_by_days(df, value_col):
        temp = df[["date", value_col]].dropna().copy()
        temp["period"] = ((temp["date"].dt.normalize() - start_date).dt.days // n_days)
        return temp.groupby("period").agg(date=("date", "min"), score=(value_col, "mean")).set_index("date")["score"]

    fig, ax = plt.subplots(figsize=(12, 5), dpi=300)

    for side, style in side_styles.items():
        side_data = analysis_df[analysis_df["side"].str.lower() == side]
        if not side_data.empty:
            aggregated = aggregate_by_days(side_data, "score")
            if not aggregated.empty:
                ax.plot(aggregated.index, aggregated.values, marker="o", markersize=5, color=style["color"], linewidth=1.5, alpha=0.6, label=style["label"])

    overall = aggregate_by_days(daily_df, "mean_score")
    if not overall.empty:
        ax.plot(overall.index, overall.values, marker="o", markersize=6, color="#2C3E50", linewidth=2.5, label="Średnia ogólna", zorder=5)
        y_min, y_max = overall.min(), overall.max()
        margin = max((y_max - y_min) * 0.3, 0.1)
        ax.set_ylim(max(0, y_min - margin), min(1.0, y_max + margin))

    ax.set_xlabel("Data", fontsize=11, fontweight="bold", color="#2C3E50")
    ax.set_ylabel("Średni Score", fontsize=11, fontweight="bold", color="#2C3E50")
    ax.set_title(title, fontsize=13, fontweight="bold", pad=15, color="#2C3E50")
    ax.xaxis.set_major_formatter(plt.matplotlib.dates.DateFormatter(date_format))
    ax.tick_params(axis="x", rotation=30)
    ax.grid(True, linestyle=":", alpha=0.6, color="#bdc3c7")
    ax.legend(frameon=True, facecolor="white", edgecolor="#bdc3c7", fontsize=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.show()

plot_aggregated_scores(n_days=1)
plot_aggregated_scores(n_days=7)

# 4. INTERAKTYWNY PULPIT WIDŻETÓW (IPYWIDGETS)
daily_factor_df = pd.merge(factors_df, daily_df, on="date", how="outer").sort_values("date").reset_index(drop=True)
daily_factor_df["interpolated_score"] = daily_factor_df["mean_score"].interpolate(method="linear").bfill().ffill()

min_date_val = daily_factor_df["date"].min()
max_date_val = daily_factor_df["date"].max()

def update_interactive_plot(selected_factor, show_present, start_date, end_date, n_days):
    if start_date > end_date:
        print("Data początkowa musi być wcześniejsza lub równa końcowej.")
        return

    df_sub = daily_factor_df[
        (daily_factor_df["date"] >= pd.Timestamp(start_date)) &
        (daily_factor_df["date"] <= pd.Timestamp(end_date))
    ].copy()

    if df_sub.empty:
        print("Brak danych w wybranym zakresie dat.")
        return

    n_days = max(1, int(n_days))
    start = pd.Timestamp(start_date)
    df_sub["period"] = ((df_sub["date"] - start).dt.days // n_days)

    agg_dict = {
        "date": ("date", "min"),
        "mean_score": ("mean_score", "mean"),
        "interpolated_score": ("interpolated_score", "mean")
    }
    for factor in FACTORS:
        if factor in df_sub.columns:
            agg_dict[str(factor)] = (factor, "max" if show_present else "min")

    aggregated = df_sub.groupby("period").agg(**agg_dict).reset_index(drop=True)

    fig, ax = plt.subplots(figsize=(12, 5), dpi=300)
    valid_scores = aggregated.dropna(subset=["mean_score"])
    ax.plot(valid_scores["date"], valid_scores["mean_score"], marker="o", markersize=5, color="#2C3E50", linewidth=2, label="Overall Mean Score", zorder=5)

    if selected_factor in aggregated.columns:
        condition_val = 1 if show_present else 0
        factor_subset = aggregated[aggregated[selected_factor] == condition_val]

        if not factor_subset.empty:
            status_label = "Present (1)" if show_present else "Absent (0)"
            ax.scatter(factor_subset["date"], factor_subset["interpolated_score"], color="#E74C3C", s=130, zorder=6, label=f"Factor '{selected_factor}' = {status_label}")

    all_scores = aggregated["interpolated_score"].dropna()
    if not all_scores.empty:
        y_min, y_max = all_scores.min(), all_scores.max()
        margin = max((y_max - y_min) * 0.25, 0.05)
        ax.set_ylim(max(0, y_min - margin), min(1.0, y_max + margin))

    ax.set_xlabel("Date", fontsize=11, fontweight="bold", color="#2C3E50")
    ax.set_ylabel("Mean Score", fontsize=11, fontweight="bold", color="#2C3E50")
    ax.set_title(f"Acne Score Evolution & Factor Impact: {selected_factor} ({n_days}-day groups)", fontsize=13, fontweight="bold", pad=15, color="#2C3E50")
    ax.tick_params(axis="x", rotation=30)
    ax.grid(True, linestyle=":", alpha=0.6, color="#bdc3c7")
    ax.legend(frameon=True, facecolor="white", edgecolor="#bdc3c7", fontsize=10)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.tight_layout()
    plt.show()

factor_dropdown = widgets.Dropdown(options=FACTORS, value=FACTORS[0] if FACTORS else None, description="Factor:", style={"description_width": "initial"})
presence_checkbox = widgets.Checkbox(value=True, description="Present (1) / Absent (0)", style={"description_width": "initial"})
start_picker = widgets.DatePicker(description="Od:", value=min_date_val.date() if pd.notna(min_date_val) else None)
end_picker = widgets.DatePicker(description="Do:", value=max_date_val.date() if pd.notna(max_date_val) else None)
n_days_slider = widgets.IntSlider(value=7, min=1, max=30, step=1, description="Grupuj co N dni:", style={"description_width": "initial"})

widgets.interact(
    update_interactive_plot,
    selected_factor=factor_dropdown,
    show_present=presence_checkbox,
    start_date=start_picker,
    end_date=end_picker,
    n_days=n_days_slider
)

# 5. ANALIZA KORELACJI SPEARMANA
factors_ddf = factors_df.copy()
scores_df = daily_df[["date", "mean_score"]].copy()
df_corr = factors_ddf.merge(scores_df, on="date", how="inner").dropna(subset=["mean_score"])

results = []
for factor in FACTORS:
    temp = df_corr[[factor, "mean_score"]].dropna()
    if temp[factor].nunique() < 2 or len(temp) < 3:
        continue
    rho, p = spearmanr(temp[factor], temp["mean_score"])
    results.append({"czynnik": factor, "rho": rho, "p_value": p, "n": len(temp)})

corr_df = pd.DataFrame(results)
if not corr_df.empty:
    corr_df["p_adj"] = multipletests(corr_df["p_value"], method="fdr_bh")[1]
    corr_df = corr_df.sort_values("rho", ascending=False)
    display(corr_df.round(3))